In [2]:
# ============================================================================
# DEBUG: Check what load_evaluation actually returns
# ============================================================================
print("Debugging load_evaluation return format...\n")

# Test with one isotope
test_iso = "nb93"
test_mt = 16

for lib in ['tendl.2019', 'endfb8.0']:
    print(f"Testing {lib}:")
    try:
        result = nuc_data.load_evaluation(test_iso, test_mt, library=lib)
        print(f"  Type: {type(result)}")
        print(f"  Is None: {result is None}")
        if result is not None:
            if isinstance(result, dict):
                print(f"  Dict keys: {result.keys()}")
                for k, v in result.items():
                    print(f"    {k}: {type(v)}")
                    if isinstance(v, pd.DataFrame):
                        print(f"      Columns: {v.columns.tolist()}")
                        print(f"      Shape: {v.shape}")
            elif isinstance(result, pd.DataFrame):
                print(f"  DataFrame columns: {result.columns.tolist()}")
                print(f"  Shape: {result.shape}")
                print(f"  Head:\n{result.head()}")
            else:
                print(f"  Value: {result}")
    except Exception as e:
        print(f"  Error: {type(e).__name__}: {e}")
    print()

Debugging load_evaluation return format...

Testing tendl.2019:
  Type: <class 'pandas.core.frame.DataFrame'>
  Is None: False
  DataFrame columns: ['Energy', 'Data']
  Shape: (64, 2)
  Head:
     Energy      Data
0  6.950693      -inf
1  6.954243 -1.711176
2  6.977724 -0.929936
3  7.000000 -0.397066
4  7.021189 -0.179806

Testing endfb8.0:
  Type: <class 'pandas.core.frame.DataFrame'>
  Is None: False
  DataFrame columns: ['Energy', 'Data']
  Shape: (60, 2)
  Head:
     Energy      Data
0  6.950803      -inf
1  6.956168 -1.966576
2  6.957080 -1.939302
3  6.959041 -1.882729
4  6.963788 -1.703335



/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/datasets.py:149: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  evaluation = pd.read_csv(path, skiprows=5, header=None, names=["Energy", "Data", "dDataLow", "dDataUpp"], delim_whitespace=True)
/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/datasets.py:149: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  evaluation = pd.read_csv(path, skiprows=5, header=None, names=["Energy", "Data", "dDataLow", "dDataUpp"], delim_whitespace=True)
/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/pandas/core/arraylike.py:399: Run

In [3]:
def overlay_exfor_evaluations(isotope=None, z=None, a=None, mt_number=16, exfordf=None, 
                               figure_dir="./figures/", libraries=['tendl.2019', 'endfb8.0'], **kwargs):
    """
    Overlay EXFOR experimental data with theoretical evaluations.
    Handles -inf values and log-space data properly.
    """
    
    # Parse isotope
    if z is None or a is None:
        if isotope is None:
            raise ValueError("Provide 'isotope' string or both 'z' and 'a'")
        z, a = _parse_isotope(isotope)
        iso_norm = isotope.lower().replace('-', '').strip()
        print(f"✓ Parsed {isotope} → Z={z}, A={a}")
    else:
        iso_norm = isotope.lower().replace('-', '').strip() if isotope else f"z{z}a{a}"
    
    # 1. Load EXFOR data
    exfor_data = exfordf[
        (exfordf['Z'] == z) & 
        (exfordf['A'] == a) & 
        (exfordf['MT'].astype(float) == float(mt_number))
    ].copy()
    
    if exfor_data.empty:
        print(f"✗ No EXFOR data for Z={z}, A={a}, MT={mt_number}")
        return None
    
    exfor_data = exfor_data[['Energy', 'Data']].copy()
    exfor_data['Energy'] = exfor_data['Energy'].astype(float)
    exfor_data['Data'] = exfor_data['Data'].astype(float)
    exfor_data = exfor_data.dropna()
    print(f"✓ Found {len(exfor_data)} EXFOR points")
    
    # 2. Load evaluations - FIXED FOR -INF VALUES
    eval_dict = {}
    for lib in libraries:
        try:
            eval_df = nuc_data.load_evaluation(iso_norm, mt_number, library=lib)
            
            if eval_df is None or eval_df.empty:
                print(f"✗ {lib}: empty or None")
                continue
            
            # Make a copy to avoid warnings
            eval_df = eval_df[['Energy', 'Data']].copy()
            
            # CRITICAL: Replace -inf with NaN BEFORE converting from log space
            eval_df['Data'] = eval_df['Data'].replace([np.inf, -np.inf], np.nan)
            
            # Convert from log10(eV) and log10(barns) to linear space
            # Only convert finite values
            eval_df['Energy'] = 10 ** eval_df['Energy'].astype(float)
            eval_df['Data'] = 10 ** eval_df['Data'].astype(float)
            
            # Remove NaN rows that resulted from -inf
            eval_df = eval_df.dropna()
            
            if eval_df.empty:
                print(f"✗ {lib}: all values were -inf")
                continue
            
            eval_dict[lib] = eval_df
            print(f"✓ Loaded {lib}: {len(eval_df)} valid points (removed {len(nuc_data.load_evaluation(iso_norm, mt_number, library=lib)) - len(eval_df)} -inf entries)")
            
        except Exception as e:
            print(f"✗ {lib}: {type(e).__name__}: {str(e)[:60]}")
            continue
    
    if not eval_dict:
        print("⚠ No evaluations loaded!")
        return None
    
    # 3. Interpolate evaluations to EXFOR energy points
    interp_dict = {}
    for lib_name, lib_data in eval_dict.items():
        interp_vals = np.interp(
            exfor_data['Energy'].values,
            lib_data['Energy'].values,
            lib_data['Data'].values,
            left=np.nan,
            right=np.nan
        )
        interp_dict[lib_name] = interp_vals
    
    # 4. Calculate error metrics
    metrics = {}
    for lib_name, interp_vals in interp_dict.items():
        valid = ~np.isnan(interp_vals)
        if valid.sum() > 0:
            exp_vals = exfor_data['Data'].values[valid]
            th_vals = interp_vals[valid]
            metrics[lib_name] = _calculate_error_metrics(exp_vals, th_vals)
    
    # 5. Create comparison dataframe
    comparison_df = exfor_data.copy()
    for lib_name, interp_vals in interp_dict.items():
        comparison_df[f'{lib_name}_pred'] = interp_vals
    
    # 6. Plotting
    plt.figure(figsize=(12, 7))
    plt.scatter(exfor_data['Energy'], exfor_data['Data'], 
                color='black', s=50, alpha=0.6, label='EXFOR', zorder=5)
    
    colors = {'tendl.2019': 'blue', 'endfb8.0': 'red', 'jeff3.3': 'green', 'jendl5.0': 'purple'}
    for lib_name, lib_data in eval_dict.items():
        color = colors.get(lib_name, 'gray')
        label = lib_name.upper()
        if lib_name in metrics:
            label += f" (RMSE={metrics[lib_name]['rmse']:.3e})"
        plt.plot(lib_data['Energy'], lib_data['Data'], 
                color=color, linewidth=2, label=label, alpha=0.8)
    
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Energy (eV)', fontsize=12)
    plt.ylabel('Cross Section (barns)', fontsize=12)
    plt.title(f'Z={z}, A={a} MT={mt_number} - EXFOR vs Evaluations', fontsize=14)
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    os.makedirs(figure_dir, exist_ok=True)
    plt.savefig(os.path.join(figure_dir, f'{iso_norm}_mt{mt_number}_overlay.png'), dpi=300)
    plt.show()
    
    return {
        'exfor': exfor_data,
        'evaluations': eval_dict,
        'interpolated': interp_dict,
        'comparison': comparison_df,
        'metrics': metrics,
        'z': z,
        'a': a
    }


# ============================================================================
# RUN WITH FIXED FUNCTION
# ============================================================================
print("Loading EXFOR data...")
exfordf = nuc_data.load_exfor()
exfordf['MT'] = exfordf['MT'].astype(int)
print(f"✓ Loaded {len(exfordf)} EXFOR records\n")

figure_dir = "./figures_exfor_eval/"
os.makedirs(figure_dir, exist_ok=True)

test_cases = [
    {'z': 41, 'a': 93, 'mt': 16, 'name': 'Nb93'},
    {'z': 79, 'a': 197, 'mt': 16, 'name': 'Au197'},
    {'z': 92, 'a': 235, 'mt': 18, 'name': 'U235'},
]

results = {}
for case in test_cases:
    print(f"\n{'='*60}")
    print(f"Processing: {case['name']} (Z={case['z']}, A={case['a']}, MT={case['mt']})")
    print(f"{'='*60}")
    
    result = overlay_exfor_evaluations(
        z=case['z'],
        a=case['a'],
        mt_number=case['mt'],
        exfordf=exfordf,
        figure_dir=figure_dir,
        libraries=['tendl.2019', 'endfb8.0']
    )
    
    if result and result['metrics']:
        results[case['name']] = result
        print(f"\n✓ Metrics for {case['name']}:")
        for lib_name, metrics in result['metrics'].items():
            print(f"  {lib_name:15s}: RMSE={metrics['rmse']:.4e}, MAPE={metrics['mape']:.2f}%")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print(f"\n\n{'='*80}")
print("SUMMARY: Error Metrics Comparison")
print(f"{'='*80}\n")

summary_data = []
for iso_name, result in results.items():
    if result['metrics']:
        for lib_name, metrics in result['metrics'].items():
            summary_data.append({
                'Isotope': iso_name,
                'Library': lib_name,
                'RMSE': f"{metrics['rmse']:.4e}",
                'MAPE (%)': f"{metrics['mape']:.2f}"
            })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
else:
    print("No metrics computed")

print(f"\n✓ All figures saved to: {figure_dir}")
print(f"✓ Processing complete!")

Loading EXFOR data...
✓ Loaded 4395295 EXFOR records


Processing: Nb93 (Z=41, A=93, MT=16)
✓ Found 372 EXFOR points
✗ tendl.2019: ValueError: too many values to unpack (expected 2)
✗ endfb8.0: ValueError: too many values to unpack (expected 2)
⚠ No evaluations loaded!

Processing: Au197 (Z=79, A=197, MT=16)
✓ Found 507 EXFOR points
✗ tendl.2019: ValueError: too many values to unpack (expected 2)
✗ endfb8.0: ValueError: too many values to unpack (expected 2)
⚠ No evaluations loaded!

Processing: U235 (Z=92, A=235, MT=18)
✓ Found 135543 EXFOR points
✗ tendl.2019: ValueError: too many values to unpack (expected 2)
✗ endfb8.0: ValueError: too many values to unpack (expected 2)
⚠ No evaluations loaded!


SUMMARY: Error Metrics Comparison

No metrics computed

✓ All figures saved to: ./figures_exfor_eval/
✓ Processing complete!


In [4]:
# ============================================================================
# DEBUG: Find unpacking error
# ============================================================================
print("Detailed debug of load_evaluation unpacking issue...\n")

test_iso = "nb93"
test_mt = 16

for lib in ['tendl.2019', 'endfb8.0']:
    print(f"\n{'='*60}")
    print(f"Testing {lib}:")
    print(f"{'='*60}")
    try:
        result = nuc_data.load_evaluation(test_iso, test_mt, library=lib)
        print(f"Type of result: {type(result)}")
        print(f"Result is None: {result is None}")
        
        if result is not None:
            # Check if it's iterable and how many items
            if isinstance(result, (tuple, list)):
                print(f"Result is tuple/list with {len(result)} items:")
                for i, item in enumerate(result):
                    print(f"  [{i}] Type: {type(item)}, Value type: {type(item).__name__}")
                    if isinstance(item, pd.DataFrame):
                        print(f"      Shape: {item.shape}, Columns: {item.columns.tolist()}")
                    elif isinstance(item, dict):
                        print(f"      Dict keys: {item.keys()}")
                    else:
                        print(f"      First 100 chars: {str(item)[:100]}")
            elif isinstance(result, pd.DataFrame):
                print(f"Result is DataFrame:")
                print(f"  Columns: {result.columns.tolist()}")
                print(f"  Shape: {result.shape}")
                print(f"  Head:\n{result.head()}")
            elif isinstance(result, dict):
                print(f"Result is dict with keys: {result.keys()}")
                for k, v in result.items():
                    print(f"  {k}: {type(v)}")
            else:
                print(f"Result value: {result}")
        
        # Try the problematic line from the original code
        print(f"\nTrying: eval_df = result[['Energy', 'Data']].copy()")
        try:
            eval_df = result[['Energy', 'Data']].copy()
            print(f"  ✓ Success! Shape: {eval_df.shape}")
        except Exception as e:
            print(f"  ✗ Error: {type(e).__name__}: {e}")
            
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

Detailed debug of load_evaluation unpacking issue...


Testing tendl.2019:
Type of result: <class 'pandas.core.frame.DataFrame'>
Result is None: False
Result is DataFrame:
  Columns: ['Energy', 'Data']
  Shape: (64, 2)
  Head:
     Energy      Data
0  6.950693      -inf
1  6.954243 -1.711176
2  6.977724 -0.929936
3  7.000000 -0.397066
4  7.021189 -0.179806

Trying: eval_df = result[['Energy', 'Data']].copy()
  ✓ Success! Shape: (64, 2)

Testing endfb8.0:
Type of result: <class 'pandas.core.frame.DataFrame'>
Result is None: False
Result is DataFrame:
  Columns: ['Energy', 'Data']
  Shape: (60, 2)
  Head:
     Energy      Data
0  6.950803      -inf
1  6.956168 -1.966576
2  6.957080 -1.939302
3  6.959041 -1.882729
4  6.963788 -1.703335

Trying: eval_df = result[['Energy', 'Data']].copy()
  ✓ Success! Shape: (60, 2)


/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/datasets.py:149: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  evaluation = pd.read_csv(path, skiprows=5, header=None, names=["Energy", "Data", "dDataLow", "dDataUpp"], delim_whitespace=True)
/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/datasets.py:149: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  evaluation = pd.read_csv(path, skiprows=5, header=None, names=["Energy", "Data", "dDataLow", "dDataUpp"], delim_whitespace=True)
/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/pandas/core/arraylike.py:399: Run

In [5]:
def overlay_exfor_evaluations(isotope=None, z=None, a=None, mt_number=16, exfordf=None, 
                               figure_dir="./figures/", libraries=['tendl.2019', 'endfb8.0'], **kwargs):
    """Overlay EXFOR with evaluations - UNPACKING FIX"""
    
    if z is None or a is None:
        if isotope is None:
            raise ValueError("Provide 'isotope' string or both 'z' and 'a'")
        z, a = _parse_isotope(isotope)
        iso_norm = isotope.lower().replace('-', '').strip()
        print(f"✓ Parsed {isotope} → Z={z}, A={a}")
    else:
        iso_norm = isotope.lower().replace('-', '').strip() if isotope else f"z{z}a{a}"
    
    # Load EXFOR data
    exfor_data = exfordf[
        (exfordf['Z'] == z) & 
        (exfordf['A'] == a) & 
        (exfordf['MT'].astype(float) == float(mt_number))
    ].copy()
    
    if exfor_data.empty:
        print(f"✗ No EXFOR data for Z={z}, A={a}, MT={mt_number}")
        return None
    
    exfor_data = exfor_data[['Energy', 'Data']].copy()
    exfor_data['Energy'] = exfor_data['Energy'].astype(float)
    exfor_data['Data'] = exfor_data['Data'].astype(float)
    exfor_data = exfor_data.dropna()
    print(f"✓ Found {len(exfor_data)} EXFOR points")
    
    # Load evaluations - FIXED UNPACKING
    eval_dict = {}
    for lib in libraries:
        try:
            eval_df = nuc_data.load_evaluation(iso_norm, mt_number, library=lib)
            if eval_df is None or eval_df.empty:
                print(f"✗ {lib}: empty or None")
                continue
            
            eval_df = eval_df[['Energy', 'Data']].copy()
            
            # FIX: Replace -inf with NaN FIRST
            eval_df['Data'] = eval_df['Data'].replace([np.inf, -np.inf], np.nan)
            
            # FIX: Simple exponentiation without .clip()
            eval_df['Energy'] = 10.0 ** eval_df['Energy'].astype(float)
            eval_df['Data'] = 10.0 ** eval_df['Data'].astype(float)
            
            # FIX: Drop NaN rows after conversion
            eval_df = eval_df.dropna()
            
            if eval_df.empty:
                print(f"✗ {lib}: all values were -inf")
                continue
            
            eval_dict[lib] = eval_df
            print(f"✓ Loaded {lib}: {len(eval_df)} valid points")
            
        except Exception as e:
            print(f"✗ {lib}: {type(e).__name__}: {str(e)[:80]}")
            import traceback
            traceback.print_exc()
    
    if not eval_dict:
        print("⚠ No evaluations loaded!")
        return None
    
    # Interpolate
    interp_dict = {}
    for lib_name, lib_data in eval_dict.items():
        interp_vals = np.interp(
            exfor_data['Energy'].values,
            lib_data['Energy'].values,
            lib_data['Data'].values,
            left=np.nan,
            right=np.nan
        )
        interp_dict[lib_name] = interp_vals
    
    # Calculate metrics
    metrics = {}
    for lib_name, interp_vals in interp_dict.items():
        valid = ~np.isnan(interp_vals)
        if valid.sum() > 0:
            exp_vals = exfor_data['Data'].values[valid]
            th_vals = interp_vals[valid]
            metrics[lib_name] = _calculate_error_metrics(exp_vals, th_vals)
    
    # Create comparison dataframe
    comparison_df = exfor_data.copy()
    for lib_name, interp_vals in interp_dict.items():
        comparison_df[f'{lib_name}_pred'] = interp_vals
    
    # Plot
    plt.figure(figsize=(12, 7))
    plt.scatter(exfor_data['Energy'], exfor_data['Data'], 
                color='black', s=50, alpha=0.6, label='EXFOR', zorder=5)
    
    colors = {'tendl.2019': 'blue', 'endfb8.0': 'red', 'jeff3.3': 'green', 'jendl5.0': 'purple'}
    for lib_name, lib_data in eval_dict.items():
        color = colors.get(lib_name, 'gray')
        label = lib_name.upper()
        if lib_name in metrics:
            label += f" (RMSE={metrics[lib_name]['rmse']:.3e})"
        plt.plot(lib_data['Energy'], lib_data['Data'], 
                color=color, linewidth=2, label=label, alpha=0.8)
    
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Energy (eV)', fontsize=12)
    plt.ylabel('Cross Section (barns)', fontsize=12)
    plt.title(f'Z={z}, A={a} MT={mt_number} - EXFOR vs Evaluations', fontsize=14)
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    os.makedirs(figure_dir, exist_ok=True)
    plt.savefig(os.path.join(figure_dir, f'{iso_norm}_mt{mt_number}_overlay.png'), dpi=300)
    plt.show()
    
    return {
        'exfor': exfor_data,
        'evaluations': eval_dict,
        'interpolated': interp_dict,
        'comparison': comparison_df,
        'metrics': metrics,
        'z': z,
        'a': a
    }


# Run it
print("Loading EXFOR data...")
exfordf = nuc_data.load_exfor()
exfordf['MT'] = exfordf['MT'].astype(int)
print(f"✓ Loaded {len(exfordf)} EXFOR records\n")

figure_dir = "./figures_exfor_eval/"
os.makedirs(figure_dir, exist_ok=True)

test_cases = [
    {'z': 41, 'a': 93, 'mt': 16, 'name': 'Nb93'},
    {'z': 79, 'a': 197, 'mt': 16, 'name': 'Au197'},
    {'z': 92, 'a': 235, 'mt': 18, 'name': 'U235'},
]

results = {}
for case in test_cases:
    print(f"\n{'='*60}")
    print(f"Processing: {case['name']}")
    print(f"{'='*60}")
    result = overlay_exfor_evaluations(z=case['z'], a=case['a'], mt_number=case['mt'], 
                                        exfordf=exfordf, figure_dir=figure_dir, 
                                        libraries=['tendl.2019', 'endfb8.0'])
    if result and result['metrics']:
        results[case['name']] = result
        for lib, m in result['metrics'].items():
            print(f"  {lib}: RMSE={m['rmse']:.4e}, MAPE={m['mape']:.2f}%")

print(f"\n{'='*80}\nSUMMARY\n{'='*80}\n")
summary_data = []
for iso, res in results.items():
    for lib, m in res['metrics'].items():
        summary_data.append({'Isotope': iso, 'Library': lib, 'RMSE': f"{m['rmse']:.4e}", 'MAPE (%)': f"{m['mape']:.2f}"})
if summary_data:
    print(pd.DataFrame(summary_data).to_string(index=False))
print(f"\n✓ Complete!")

Loading EXFOR data...
✓ Loaded 4395295 EXFOR records


Processing: Nb93
✓ Found 372 EXFOR points
✗ tendl.2019: ValueError: too many values to unpack (expected 2)
✗ endfb8.0: ValueError: too many values to unpack (expected 2)
⚠ No evaluations loaded!

Processing: Au197
✓ Found 507 EXFOR points
✗ tendl.2019: ValueError: too many values to unpack (expected 2)
✗ endfb8.0: ValueError: too many values to unpack (expected 2)
⚠ No evaluations loaded!

Processing: U235
✓ Found 135543 EXFOR points
✗ tendl.2019: ValueError: too many values to unpack (expected 2)
✗ endfb8.0: ValueError: too many values to unpack (expected 2)
⚠ No evaluations loaded!

SUMMARY


✓ Complete!


Traceback (most recent call last):
  File "/var/folders/1l/tff5w1ks3pz74mb34g35c7v00000gn/T/ipykernel_34550/2033152788.py", line 35, in overlay_exfor_evaluations
    eval_df = nuc_data.load_evaluation(iso_norm, mt_number, library=lib)
  File "/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/datasets.py", line 137, in load_evaluation
    isotope = gen_utils.parse_isotope(isotope, parse_for='endf')
  File "/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/general_utilities.py", line 149, in parse_isotope
    element, mass = re.findall(r'[A-Za-z]+|\d+', isotope)
ValueError: too many values to unpack (expected 2)
Traceback (most recent call last):
  File "/var/folders/1l/tff5w1ks3pz74mb34g35c7v00000gn/T/ipykernel_34550/2033152788.py", line 35, in overlay_exfor_evaluations
    eval_df = nuc_data.load_evaluation(iso_norm, mt_number, library=lib)
  File "/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/datasets.py", line 137, in load_evaluatio

In [ ]:

    # # 2. Load evaluations
    # eval_dict = {}
    # for lib in libraries:
    #     try:
    #         eval_df = nuc_data.load_evaluation(iso_norm, mt_number, library=lib)
    #         if eval_df is not None and not eval_df.empty:
    #             # Select only Energy and Data columns
    #             eval_df = eval_df[['Energy', 'Data']].copy()
    #             # Convert from log10(eV) and log10(barns) to linear space
    #             eval_df['Energy'] = 10 ** eval_df['Energy']
    #             eval_df['Data'] = 10 ** eval_df['Data'].clip(lower=1e-15)
    #             eval_dict[lib] = eval_df.dropna()
    #             print(f"✓ Loaded {lib}: {len(eval_df)} points")
    #     except Exception as e:
    #         print(f"✗ Could not load {lib}: {str(e)[:60]}")
    
    # if not eval_dict:
    #     print("No evaluations loaded!")
    #     return None
    # 2. Load evaluations

In [9]:
def overlay_exfor_evaluations(isotope=None, z=None, a=None, mt_number=16, exfordf=None, 
                               figure_dir="./figures/", libraries=['tendl.2019', 'endfb8.0'], **kwargs):
    """
    Overlay EXFOR experimental data with theoretical evaluations.
    
    Parameters:
    -----------
    isotope : str, optional
        Isotope name (e.g., "Nb93", "Au197", "U235")
    z : int, optional
        Atomic number
    a : int, optional
        Mass number
    mt_number : int
        MT reaction number (16 for n,2n, 18 for fission)
    exfordf : pd.DataFrame
        EXFOR dataframe
    figure_dir : str
        Save directory
    libraries : list
        Libraries to load
    
    Returns:
    --------
    dict with keys: 'exfor', 'evaluations', 'interpolated', 'comparison', 'metrics', 'z', 'a'
    """
    
    # Parse isotope
    if z is None or a is None:
        if isotope is None:
            raise ValueError("Provide 'isotope' string or both 'z' and 'a'")
        z, a = _parse_isotope(isotope)
        iso_norm = isotope.lower().replace('-', '').strip()
        print(f"✓ Parsed {isotope} → Z={z}, A={a}")
    else:
        iso_norm = isotope.lower().replace('-', '').strip() if isotope else f"z{z}a{a}"
    
    # 1. Load EXFOR data
    exfor_data = exfordf[
        (exfordf['Z'] == z) & 
        (exfordf['A'] == a) & 
        (exfordf['MT'].astype(float) == float(mt_number))
    ].copy()
    
    if exfor_data.empty:
        print(f"✗ No EXFOR data for Z={z}, A={a}, MT={mt_number}")
        return None
    
    exfor_data = exfor_data[['Energy', 'Data']].copy()
    exfor_data['Energy'] = exfor_data['Energy'].astype(float)
    exfor_data['Data'] = exfor_data['Data'].astype(float)
    exfor_data = exfor_data.dropna()
    print(f"✓ Found {len(exfor_data)} EXFOR points")
    
    # 2. Load evaluations - VERBOSE DEBUG VERSION
    eval_dict = {}
    for lib in libraries:
        try:
            print(f"\n  → Loading {lib}...")
            eval_df = nuc_data.load_evaluation(iso_norm, mt_number, library=lib)
            print(f"    ✓ load_evaluation returned: type={type(eval_df)}, is_None={eval_df is None}")
        
            if eval_df is not None and not eval_df.empty:
                print(f"    ✓ DataFrame is not None/empty, shape={eval_df.shape}")
            
                # Select only Energy and Data columns
                eval_df = eval_df[['Energy', 'Data']].copy()
                print(f"    ✓ Selected columns, shape={eval_df.shape}")
            
                # Convert from log10(eV) and log10(barns) to linear space
                print(f"    → Converting Energy from log10...")
                eval_df['Energy'] = 10 ** eval_df['Energy']
                print(f"    ✓ Energy converted")
            
                print(f"    → Converting Data from log10 with .clip()...")
                print(f"      Data before clip: min={eval_df['Data'].min()}, max={eval_df['Data'].max()}")
                clipped = eval_df['Data'].clip(lower=1e-15)
                print(f"      Data after clip: min={clipped.min()}, max={clipped.max()}")
                converted = 10 ** clipped
                print(f"      Data after 10**: min={converted.min()}, max={converted.max()}")
                eval_df['Data'] = converted
                print(f"    ✓ Data converted")
            
                print(f"    → Dropping NaN values...")
                before_drop = len(eval_df)
                eval_df = eval_df.dropna()
                after_drop = len(eval_df)
                print(f"    ✓ Dropped {before_drop - after_drop} rows, remaining={after_drop}")
            
                eval_dict[lib] = eval_df
                print(f"    ✓ Added to eval_dict. Total libraries loaded: {len(eval_dict)}")
                print(f"✓ Loaded {lib}: {len(eval_df)} points")
            else:
                print(f"    ✗ DataFrame is None or empty: is_None={eval_df is None}, empty={eval_df.empty if eval_df is not None else 'N/A'}")
            
        except Exception as e:
            print(f"    ✗ EXCEPTION: {type(e).__name__}: {e}")
            import traceback
            traceback.print_exc()

    print(f"\n  Final eval_dict status:")
    print(f"    eval_dict length: {len(eval_dict)}")
    print(f"    eval_dict keys: {eval_dict.keys()}")

    if not eval_dict:
        print("⚠ No evaluations loaded!")
        return None
    
    print(f"✓ Successfully loaded {len(eval_dict)} evaluation libraries")
    
    # 3. Interpolate evaluations to EXFOR energy points
    interp_dict = {}
    for lib_name, lib_data in eval_dict.items():
        interp_vals = np.interp(
            exfor_data['Energy'].values,
            lib_data['Energy'].values,
            lib_data['Data'].values,
            left=np.nan,
            right=np.nan
        )
        interp_dict[lib_name] = interp_vals
    
    # 4. Calculate error metrics
    metrics = {}
    for lib_name, interp_vals in interp_dict.items():
        valid = ~np.isnan(interp_vals)
        if valid.sum() > 0:
            exp_vals = exfor_data['Data'].values[valid]
            th_vals = interp_vals[valid]
            metrics[lib_name] = _calculate_error_metrics(exp_vals, th_vals)
    
    # 5. Create comparison dataframe
    comparison_df = exfor_data.copy()
    for lib_name, interp_vals in interp_dict.items():
        comparison_df[f'{lib_name}_pred'] = interp_vals
    
    # 6. Plotting
    plt.figure(figsize=(12, 7))
    plt.scatter(exfor_data['Energy'], exfor_data['Data'], 
                color='black', s=50, alpha=0.6, label='EXFOR', zorder=5)
    
    colors = {'tendl.2019': 'blue', 'endfb8.0': 'red', 'jeff3.3': 'green', 'jendl5.0': 'purple'}
    for lib_name, lib_data in eval_dict.items():
        color = colors.get(lib_name, 'gray')
        label = lib_name.upper()
        if lib_name in metrics:
            label += f" (RMSE={metrics[lib_name]['rmse']:.3e})"
        plt.plot(lib_data['Energy'], lib_data['Data'], 
                color=color, linewidth=2, label=label, alpha=0.8)
    
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Energy (eV)', fontsize=12)
    plt.ylabel('Cross Section (barns)', fontsize=12)
    plt.title(f'Z={z}, A={a} MT={mt_number} - EXFOR vs Evaluations', fontsize=14)
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    os.makedirs(figure_dir, exist_ok=True)
    plt.savefig(os.path.join(figure_dir, f'{iso_norm}_mt{mt_number}_overlay.png'), dpi=300)
    plt.show()
    
    return {
        'exfor': exfor_data,
        'evaluations': eval_dict,
        'interpolated': interp_dict,
        'comparison': comparison_df,
        'metrics': metrics,
        'z': z,
        'a': a
    }

In [10]:
# ============================================================================
# RUN WITH VERBOSE DEBUG FUNCTION
# ============================================================================
print("Loading EXFOR data...")
exfordf = nuc_data.load_exfor()
exfordf['MT'] = exfordf['MT'].astype(int)
print(f"✓ Loaded {len(exfordf)} EXFOR records\n")

figure_dir = "./figures_exfor_eval/"
os.makedirs(figure_dir, exist_ok=True)

test_cases = [
    {'z': 41, 'a': 93, 'mt': 16, 'name': 'Nb93'},
    {'z': 79, 'a': 197, 'mt': 16, 'name': 'Au197'},
    {'z': 92, 'a': 235, 'mt': 18, 'name': 'U235'},
]

results = {}
for case in test_cases:
    print(f"\n{'='*60}")
    print(f"Processing: {case['name']} (Z={case['z']}, A={case['a']}, MT={case['mt']})")
    print(f"{'='*60}")
    
    result = overlay_exfor_evaluations(
        z=case['z'],
        a=case['a'],
        mt_number=case['mt'],
        exfordf=exfordf,
        figure_dir=figure_dir,
        libraries=['tendl.2019', 'endfb8.0']
    )
    
    if result and result['metrics']:
        results[case['name']] = result
        print(f"\n✓ Metrics for {case['name']}:")
        for lib_name, metrics in result['metrics'].items():
            print(f"  {lib_name:15s}: RMSE={metrics['rmse']:.4e}, MAPE={metrics['mape']:.2f}%")
    else:
        print(f"✗ No results for {case['name']}")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print(f"\n\n{'='*80}")
print("SUMMARY: Error Metrics Comparison")
print(f"{'='*80}\n")

summary_data = []
for iso_name, result in results.items():
    if result['metrics']:
        for lib_name, metrics in result['metrics'].items():
            summary_data.append({
                'Isotope': iso_name,
                'Library': lib_name,
                'RMSE': f"{metrics['rmse']:.4e}",
                'MAPE (%)': f"{metrics['mape']:.2f}"
            })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
else:
    print("⚠ No metrics computed - check debug output above")

print(f"\n✓ All figures saved to: {figure_dir}")
print(f"✓ Processing complete!")

Loading EXFOR data...
✓ Loaded 4395295 EXFOR records


Processing: Nb93 (Z=41, A=93, MT=16)
✓ Found 372 EXFOR points

  → Loading tendl.2019...
    ✗ EXCEPTION: ValueError: too many values to unpack (expected 2)

  → Loading endfb8.0...
    ✗ EXCEPTION: ValueError: too many values to unpack (expected 2)

  Final eval_dict status:
    eval_dict length: 0
    eval_dict keys: dict_keys([])
⚠ No evaluations loaded!
✗ No results for Nb93

Processing: Au197 (Z=79, A=197, MT=16)
✓ Found 507 EXFOR points

  → Loading tendl.2019...
    ✗ EXCEPTION: ValueError: too many values to unpack (expected 2)

  → Loading endfb8.0...
    ✗ EXCEPTION: ValueError: too many values to unpack (expected 2)

  Final eval_dict status:
    eval_dict length: 0
    eval_dict keys: dict_keys([])
⚠ No evaluations loaded!
✗ No results for Au197

Processing: U235 (Z=92, A=235, MT=18)
✓ Found 135543 EXFOR points

  → Loading tendl.2019...
    ✗ EXCEPTION: ValueError: too many values to unpack (expected 2)

  → Loading en

Traceback (most recent call last):
  File "/var/folders/1l/tff5w1ks3pz74mb34g35c7v00000gn/T/ipykernel_34550/1825821816.py", line 60, in overlay_exfor_evaluations
    eval_df = nuc_data.load_evaluation(iso_norm, mt_number, library=lib)
  File "/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/datasets.py", line 137, in load_evaluation
    isotope = gen_utils.parse_isotope(isotope, parse_for='endf')
  File "/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/general_utilities.py", line 149, in parse_isotope
    element, mass = re.findall(r'[A-Za-z]+|\d+', isotope)
ValueError: too many values to unpack (expected 2)
Traceback (most recent call last):
  File "/var/folders/1l/tff5w1ks3pz74mb34g35c7v00000gn/T/ipykernel_34550/1825821816.py", line 60, in overlay_exfor_evaluations
    eval_df = nuc_data.load_evaluation(iso_norm, mt_number, library=lib)
  File "/opt/anaconda3/envs/ml_msci_v2/lib/python3.10/site-packages/nucml/datasets.py", line 137, in load_evaluatio